In [ ]:
#Donwloading dataset
import requests
import tarfile
import os

output_dir = '../data'
os.makedirs(output_dir, exist_ok=True)

url = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"

tar_path = os.path.join(output_dir, 'aclImdb_v1.tar.gz')

response = requests.get(url, stream=True)
if response.status_code  == 200:
    with open(tar_path, 'wb') as file:
        for chunk in response.iter_content(chunk_size=8192):
            file.write(chunk)
    print(f'Downloaded file: {tar_path}')
else:
    print(f'Donwloaded error {response.status_code}')
    
with tarfile.open(tar_path, 'r:gz') as tar_ref:
    tar_ref.extractall(output_dir)
    print(f'Extracted files in: {output_dir}')
    
imdb_dir = os.path.join(output_dir, 'aclImdb')
if os.path.exists(imdb_dir):
    for item in os.listdir(imdb_dir):
        print(f' - {item}')
else:
    print('Error: The folder aclImdb not found')

In [ ]:
import os, pathlib, shutil, random

base_dir = pathlib.Path('../data/aclImdb')
val_dir = base_dir/'val'
train_dir = base_dir/'train'
for category in ('neg', 'pos'):
    os.makedirs(val_dir/category)
    files = os.listdir(train_dir/category)
    random.Random(1337).shuffle(files)
    num_val_samples = int(0.2*len(files))
    val_files = files[-num_val_samples:]
    for fname in val_files:
        shutil.move(train_dir/category/fname,
                    val_dir/category/fname)


In [1]:
from tensorflow import keras
batch_size = 32

train_ds = keras.utils.text_dataset_from_directory(
    '../data/aclImdb/train', batch_size=batch_size
)
val_ds = keras.utils.text_dataset_from_directory(
    '../data/aclImdb/val', batch_size=batch_size
)
test_ds = keras.utils.text_dataset_from_directory(
    '../data/aclImdb/test', batch_size=batch_size
)

Found 20000 files belonging to 2 classes.
Found 5000 files belonging to 2 classes.
Found 25000 files belonging to 2 classes.


In [2]:
for inputs, targets in train_ds:
    print('inputs.shape', inputs.shape)
    print('inputs.dtype', inputs.dtype)
    print('targets.shape', targets.shape)
    print('targets.dtype', targets.dtype)
    print('inputs[0]', inputs[0])
    print('targets[0]', targets[0])
    break

inputs.shape (32,)
inputs.dtype <dtype: 'string'>
targets.shape (32,)
targets.dtype <dtype: 'int32'>
inputs[0] tf.Tensor(b'This film\'s premise is so simple and obvious that only a Texas millionaire high on oil fumes and whiskey would have a problem understanding it if someone shouted it across the proverbial parking lot. In summary: the oil business is in cahoots with The Government (or Gummint if you prefer), the Gummint is in cahoots with Middle Eastern despots, and the CIA is a singular festering pool of double dealing sons-of-(insert word) willing to toe any line that comes their way. The only people that get done over are the good ones, like Mr Clooney ("Bob"). Oh, and terrorism is a result of the poverty which globalization creates when wicked multinationals stalk the world looking for a tasty takeover or three . That really fits to the profiles of the well-heeled 9/11 perpetrators.<br /><br />In Syriana this facile tissue of political half-truths and Hollywood holograms is stir

In [3]:
#Processing the dataset with textVectorization and multi-hot
from tensorflow.keras import layers

text_vectorization = layers.TextVectorization (
    max_tokens=20000,
    output_mode='multi_hot'
)
text_only_train_ds  = train_ds.map(lambda x, y: x)
text_vectorization.adapt(text_only_train_ds)

binary_1gram_train_ds = train_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)
binary_1gram_val_ds = val_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)
binary_1gram_test_ds = test_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)

In [4]:
for inputs, targets in binary_1gram_train_ds:
    print("inputs.shape:", inputs.shape)
    print("inputs.dtype:", inputs.dtype)
    print("targets.shape:", targets.shape)
    print("targets.dtype:", targets.dtype)
    print("inputs[0]:", inputs[0])
    print("targets[0]:", targets[0])
    break

inputs.shape: (32, 20000)
inputs.dtype: <dtype: 'int64'>
targets.shape: (32,)
targets.dtype: <dtype: 'int32'>
inputs[0]: tf.Tensor([0 1 1 ... 0 0 0], shape=(20000,), dtype=int64)
targets[0]: tf.Tensor(0, shape=(), dtype=int32)


In [5]:
from tensorflow import keras

def get_model(max_tokens=20000, hidden_dim=16):
    inputs = keras.Input(shape=(max_tokens,))
    x = layers.Dense(hidden_dim, activation='relu')(inputs)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)
    model = keras.Model(inputs, outputs)
    model.compile(optimizer='rmsprop',
                loss='binary_crossentropy',
                metrics=['accuracy'])
    return model

In [6]:
model = get_model()
model.summary()
callbacks = [
    keras.callbacks.ModelCheckpoint('binary_1gram.keras',
                                    save_best_only=True)
]
model.fit(binary_1gram_train_ds.cache(),
        validation_data=binary_1gram_val_ds.cache(),
        epochs=10,
        callbacks=callbacks)
model = keras.models.load_model('binary_1gram.keras')
print(f'Test acc: {model.evaluate(binary_1gram_test_ds)[1]:.3f}')

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 20000)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │       320,016 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 320,033 (1.22 MB)

 Trainable params: 320,033 (1.22 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 28s 43ms/step - accuracy: 0.8381 - loss: 0.3939 - val_accuracy: 0.8790 - val_loss: 0.3000
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9025 - loss: 0.2631 - val_accuracy: 0.8828 - val_loss: 0.3071
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9162 - loss: 0.2379 - val_accuracy: 0.8844 - val_loss: 0.3277
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9252 - loss: 0.2176 - val_accuracy: 0.8798 - val_loss: 0.3393
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9303 - loss: 0.2064 - val_accuracy: 0.8814 - val_loss: 0.3575
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9341 - loss: 0.2020 - val_accuracy: 0.8818 - val_loss: 0.3703
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9367 - loss: 0.2029 - val_accuracy: 0.8814 - val_loss: 0.3900
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9370 - loss: 0.1998 - val_accuracy: 

In [7]:
#Using n-grams to vectorize the data gives more context to the model improving the accuracy
text_vectorization = layers.TextVectorization(
    ngrams=2, 
    max_tokens=20000,
    output_mode='multi_hot',
)

text_vectorization.adapt(text_only_train_ds)
binary_2gram_train_ds = train_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)
binary_2gram_val_ds = val_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)
binary_2gram_test_ds = test_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)

model = get_model()
model.summary()
callbacks = [
    keras.callbacks.ModelCheckpoint('binary_2gram.keras',
                                    save_best_only=True)
]
model.fit(binary_2gram_train_ds.cache(),
        validation_data=binary_2gram_val_ds.cache(),
        epochs=10,
        callbacks=callbacks)
model = keras.models.load_model('binary_2gram.keras')
print(f'Test acc: {model.evaluate(binary_2gram_test_ds)[1]:.3f}')

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 20000)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │       320,016 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 320,033 (1.22 MB)

 Trainable params: 320,033 (1.22 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 7s 10ms/step - accuracy: 0.8501 - loss: 0.3676 - val_accuracy: 0.8906 - val_loss: 0.2801
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9184 - loss: 0.2344 - val_accuracy: 0.8906 - val_loss: 0.2943
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9372 - loss: 0.2018 - val_accuracy: 0.8910 - val_loss: 0.3094
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9445 - loss: 0.1829 - val_accuracy: 0.8872 - val_loss: 0.3290
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9503 - loss: 0.1714 - val_accuracy: 0.8884 - val_loss: 0.3459
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9514 - loss: 0.1677 - val_accuracy: 0.8850 - val_loss: 0.3604
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9557 - loss: 0.1572 - val_accuracy: 0.8874 - val_loss: 0.3839
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9586 - loss: 0.1572 - val_accuracy: 0

In [8]:
for inputs, targets in binary_2gram_train_ds:
    print(inputs[0])
    print(targets[0])
    break

tf.Tensor([1 1 1 ... 0 0 0], shape=(20000,), dtype=int64)
tf.Tensor(1, shape=(), dtype=int32)


In [9]:
#TF-IDF measure the frecuency of any term in the document, giving more importance to the less frequent ones

text_vectorization = layers.TextVectorization(
    ngrams=2,
    max_tokens=20000,
    output_mode='tf_idf'
)

In [10]:
text_vectorization.adapt(text_only_train_ds)

tfidf_2gram_train_ds = train_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)
tfidf_2gram_val_ds = val_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)
tfidf_2gram_test_ds = test_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)

model = get_model()
model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 20000)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 16)             │       320,016 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 320,033 (1.22 MB)

 Trainable params: 320,033 (1.22 MB)

 Non-trainable params: 0 (0.00 B)

In [11]:
for inputs, targets in tfidf_2gram_train_ds:
    print(inputs[0])
    print(targets[0])
    break

tf.Tensor([549.4636    10.464026   3.555597 ...   0.         0.         0.      ], shape=(20000,), dtype=float32)
tf.Tensor(0, shape=(), dtype=int32)


In [12]:
callbacks = [
    keras.callbacks.ModelCheckpoint('tfidf_2gram.keras',
                                    save_best_only=True)
]
model.fit(tfidf_2gram_train_ds.cache(),
        validation_data=tfidf_2gram_val_ds.cache(),
        epochs=10,
        callbacks=callbacks)
model = keras.models.load_model('tfidf_2gram.keras')
print(f'Test acc: {model.evaluate(tfidf_2gram_test_ds)[1]:.3f}')

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 20s 31ms/step - accuracy: 0.7855 - loss: 0.4727 - val_accuracy: 0.8926 - val_loss: 0.2889
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8749 - loss: 0.3183 - val_accuracy: 0.8328 - val_loss: 0.4725
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8868 - loss: 0.2848 - val_accuracy: 0.8860 - val_loss: 0.3086
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8934 - loss: 0.2641 - val_accuracy: 0.8814 - val_loss: 0.3263
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8952 - loss: 0.2568 - val_accuracy: 0.8758 - val_loss: 0.3414
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8988 - loss: 0.2480 - val_accuracy: 0.8660 - val_loss: 0.3354
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9020 - loss: 0.2434 - val_accuracy: 0.8668 - val_loss: 0.3601
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9071 - loss: 0.2259 - val_accuracy: 

In [13]:
from tensorflow.keras import layers

max_length = 600
max_tokens = 20000
text_vectorization = layers.TextVectorization(
    max_tokens=max_tokens,
    output_mode='int',
    output_sequence_length=max_length,
)
text_vectorization.adapt(text_only_train_ds)

int_train_ds = train_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)
int_val_ds = val_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)
int_test_ds = test_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4) 

In [14]:
for inputs, targets in int_train_ds:
    print(inputs[0])
    print(targets[0])
    break

tf.Tensor(
[   10    67   110   548     5    11    20   343    10   367   638     9
    33  1731    52 11626   928    10   207     9    67  1757   105   377
   821  6110    53  3057 18255    21    86    73     9 11748     5   258
    11   927    10    67     6    66     9    10    14  3626   753    15
  8782     7   155   238     4    50    18    10   283     9    17     4
   590     5   337     3    72  1429    51    71    72    67    31   105
  1081   526  5327   351  5665  1230    11     7    34  1573    20     3
    15   135  2097    30   869   549   387     3  4453   199     2   628
   351     1    10 10263  1742    12  1802   646     5    11    20    76
  9973    53    73    21    65   591     5   119    44   792   110  1860
    41     1    73    23   468    75    11    18    80    23   468    26
  1789    85    39   128   102    44    23    26   467  2378   105     5
     2  1174    12   956   131     8  1669  1707     2   280    14    79
    50    15     9   117    22  1833   3

In [15]:
import tensorflow as tf

inputs = keras.Input(shape=(None,), dtype='int64')
embedded = layers.Lambda(
    lambda x: tf.one_hot(x, depth=max_tokens),
    output_shape=(None, max_tokens) 
)(inputs)
x = layers.Bidirectional(layers.LSTM(32))(embedded)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation='sigmoid')(x)
model = keras.Model(inputs, outputs)
model.compile(optimizer='rmsprop',
            loss='binary_crossentropy',
            metrics=['accuracy'])
model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, None)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda (Lambda)                 │ (None, None, 20000)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 64)             │     5,128,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,128,513 (19.56 MB)

 Trainable params: 5,128,513 (19.56 MB)

 Non-trainable params: 0 (0.00 B)

In [16]:
callbacks = [
    keras.callbacks.ModelCheckpoint('one_hot_bidir_lstm.keras',
                                    save_best_only=True)
]
model.fit(int_train_ds, validation_data=int_val_ds, epochs=10,
          callbacks=callbacks)
model = keras.models.load_model('one_hot_bidir_lstm.keras')
print(f'Test acc: {model.evaluate(int_test_ds)[1]:.3f}')

Epoch 1/10
115/625 ━━━━━━━━━━━━━━━━━━━━ 51:44 6s/step - accuracy: 0.5055 - loss: 0.6933


KeyboardInterrupt

